In [14]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter


In [15]:
df = pd.read_csv("test_without_labels.csv", keep_default_na=False)
df = df.drop(columns=["Usage"])
df


,Text
0,Hüttwilen el xe on comune del Canton Turgovia ...
1,La leĝo zorgas pri kompenso de nur la plej gra...
2,پک اپ پر اپنے ڈرائیور سے پہلے پہنچیں
3,Mukmu Ch'itana mukmu icha Butun nisqaqa nisqa...
4,Iwe lon ena fansoun lupwen ra aleani än Mo...
...,...
190562,Seksyon fin travèse à portant paske tout aran...
190563,ἐξῆλθεν δὲ εἰς ⸀Ταρσὸν ἀναζητῆσαι Σαῦλον
190564,Mi yigo’o sugnagrad e Kan Nthothup miyad tabab...
190565,En dellos ensayos yá teo camentao esos momento...


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
!nvidia-smi

cuda
Tue Feb 25 21:50:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.36                 Driver Version: 566.36         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   34C    P8              1W /  138W |    7209MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [17]:
#PREPROCESSING PART

import re
import jieba

def remove_urls(text):
    """Remove URLs do texto."""
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, '', text)

def contains_chinese(text):
    """Verifica se o texto contém caracteres chineses."""
    return any('\u4e00' <= ch <= '\u9fff' for ch in text)

def tokenize(text):
    """Tokeniza o texto, usando Jieba para chinês e split para outras línguas."""
    text = remove_urls(text).lower().strip()
    if contains_chinese(text):
        tokens = list(jieba.cut(text))
    else:
        tokens = text.split()
    return [token for token in tokens if token] 


def build_vocab(tokenized_texts, min_freq=1):
    """Builds a vocabulary mapping token->index."""
    counter = Counter(token for tokens in tokenized_texts for token in tokens)
    vocab = {token for token, freq in counter.items() if freq >= min_freq}
    # Reserve special tokens:
    vocab = {"<pad>", "<unk>"} | vocab
    token2idx = {token: idx for idx, token in enumerate(sorted(vocab))}
    idx2token = {idx: token for token, idx in token2idx.items()}
    return token2idx, idx2token

def numericalize(tokens, token2idx):
    """Converts a list of tokens to a list of indices."""
    return [token2idx.get(token, token2idx["<unk>"]) for token in tokens]

def pad_sequence(seq, max_length, pad_value):
    """Pads a sequence to max_length."""
    return seq + [pad_value] * (max_length - len(seq))


In [18]:
class LanguageDataset(Dataset):
    def __init__(self, df, input_token2idx, output_token2idx, max_input_len=50, max_output_len=3):
        """
        df: DataFrame with columns: ID, Text, Label.
        For output, we will create a sequence like: [<sos>, language_token, <eos>]
        """
        self.data = df
        self.input_token2idx = input_token2idx
        self.output_token2idx = output_token2idx
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len  # expected to be 3 (sos, token, eos)
        self.pad_idx = input_token2idx["<pad>"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Process input text
        tokens = tokenize(row["Text"])
        num_tokens = numericalize(tokens, self.input_token2idx)
        if len(num_tokens) > self.max_input_len:
            num_tokens = num_tokens[:self.max_input_len]
        else:
            num_tokens = pad_sequence(num_tokens, self.max_input_len, self.pad_idx)
        input_tensor = torch.tensor(num_tokens, dtype=torch.long)

        # Process output label: create sequence [<sos>, label, <eos>]
        label = row["Label"].lower()  # assuming label is like "en", "fr", etc.
        output_seq = [self.output_token2idx["<sos>"], self.output_token2idx.get(label, self.output_token2idx["<unk>"]), self.output_token2idx["<eos>"]]
        # If needed pad output sequence (here we assume fixed length = 3)
        if len(output_seq) < self.max_output_len:
            output_seq = pad_sequence(output_seq, self.max_output_len, self.output_token2idx["<pad>"])
        output_tensor = torch.tensor(output_seq, dtype=torch.long)

        return {"input": input_tensor, "target": output_tensor}

In [20]:
#Seq2Seg method
class Encoder(nn.Module):
    def __init__(self, input_vocab_size, embed_size, hidden_size, num_layers=1):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)

    def forward(self, src):
        # src: (batch, seq_len)
        embedded = self.embedding(src)  # (batch, seq_len, embed_size)
        outputs, (hidden, cell) = self.lstm(embedded)  # hidden: (num_layers, batch, hidden_size)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_vocab_size, embed_size, hidden_size, num_layers=1):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_vocab_size)

    def forward(self, input_token, hidden, cell):
        # input_token: (batch, 1)
        embedded = self.embedding(input_token)  # (batch, 1, embed_size)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))  # (batch, output_vocab_size)
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        src: (batch, src_len)
        trg: (batch, trg_len) with trg[:,0] as <sos>
        """
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.fc.out_features

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Encode input sequence
        hidden, cell = self.encoder(src)

        # First input to decoder is the <sos> token
        input_token = trg[:, 0].unsqueeze(1)  # (batch, 1)

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = output
            # Decide if we are going to use teacher forcing
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input_token = trg[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

In [21]:
def train(model, dataloader, optimizer, criterion, clip, device, print_debug=False):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        inputs, targets = batch["input"].to(device), batch["target"].to(device)
        optimizer.zero_grad()

        # Forward pass through the seq2seq model.
        outputs = model(inputs, targets)  # outputs shape: (batch, trg_len, vocab_size)

        # Extract the valid outputs and targets (middle token, index 1)
        valid_output = outputs[:, 1, :]  # shape: (batch, vocab_size)
        valid_target = targets[:, 1]     # shape: (batch,)

        # Print debug information for the first batch
        if print_debug and batch_idx == 0:
            print("=== Debug Information ===")
            print("Input batch shape:", inputs.shape)
            print("Target batch shape:", targets.shape)
            print("Valid output shape (for label token):", valid_output.shape)
            print("Valid target shape:", valid_target.shape)
            print("Sample valid output logits (first instance):", valid_output[0].detach().cpu().numpy())
            print("Sample valid target (first instance):", valid_target[0].detach().cpu().item())
            print("=========================")

        loss = criterion(valid_output, valid_target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [24]:
# Hyperparameters
BATCH_SIZE = 128
EMBED_SIZE = 500
HIDDEN_SIZE = 256
NUM_LAYERS = 2
N_EPOCHS = 1000
CLIP = 1.0
MAX_INPUT_LEN = 50
MAX_OUTPUT_LEN = 3  # <sos>, token, <eos>


df = pd.read_csv("train_submission.csv", keep_default_na=False)
df = df.drop(columns=["Usage"])

# Tokenize input texts and build vocabulary
tokenized_texts = [tokenize(text) for text in df["Text"]]
input_token2idx, input_idx2token = build_vocab(tokenized_texts)

output_tokens = {"<pad>", "<unk>", "<sos>", "<eos>"}
# Assume labels are strings like "en", "fr", etc.
output_tokens |= set(df["Label"].str.lower().unique())
output_token2idx = {token: idx for idx, token in enumerate(sorted(output_tokens))}
output_idx2token = {idx: token for token, idx in output_token2idx.items()}



In [25]:


''' # Split the data into train, validation, test (using stratified split if possible)
train_val_df, test_df = train_test_split(df, test_size=0.1, stratify=df["Label"], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1, stratify=train_val_df["Label"], random_state=42)

# Create Dataset objects
train_dataset = LanguageDataset(train_df, input_token2idx, output_token2idx, MAX_INPUT_LEN, MAX_OUTPUT_LEN)
val_dataset   = LanguageDataset(val_df, input_token2idx, output_token2idx, MAX_INPUT_LEN, MAX_OUTPUT_LEN)
test_dataset  = LanguageDataset(test_df, input_token2idx, output_token2idx, MAX_INPUT_LEN, MAX_OUTPUT_LEN)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE) '''


# Create Dataset objects
train_dataset = LanguageDataset(df, input_token2idx, output_token2idx, MAX_INPUT_LEN, MAX_OUTPUT_LEN)


# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

#train_labels = train_df["Label"].str.lower().values
train_labels = df["Label"].str.lower().values
classes = np.unique(train_labels)
class_weights = compute_class_weight('balanced', classes=classes, y=train_labels)
# Map these weights to the corresponding index in the output vocabulary:
weight_tensor = torch.ones(len(output_token2idx))
for i, lang in enumerate(classes):
    if lang in output_token2idx:
        weight_tensor[output_token2idx[lang]] = class_weights[i]
# Note: We typically ignore the weight for <pad>, <sos>, and <eos>.

# Initialize the model
encoder = Encoder(input_vocab_size=len(input_token2idx), embed_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS)
decoder = Decoder(output_vocab_size=len(output_token2idx), embed_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS)
model = Seq2Seq(encoder, decoder, device).to(device)

# Define optimizer and loss function. We use CrossEntropyLoss ignoring the <pad> token (index of <pad> in output vocabulary).
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(weight=weight_tensor.to(device), ignore_index=output_token2idx["<pad>"])

best_loss = float("inf")

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion, CLIP, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.3f}")

    # Save model if it achieves a higher accuracy
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save(model.state_dict(), "best_seq2seq_model.pt")
        print(f"New best model saved with loss: {best_loss:.3f}")


cuda


In [ ]:
#Create te prediction

test_df = pd.read_csv("test_without_labels.csv", keep_default_na=False)
model.load_state_dict(torch.load("best_seq2seq_model.pt"))
model.eval()  # Set model to evaluation mode
test_texts = test_df["Text"].apply(lambda x: numericalize(tokenize(x), input_token2idx))

test_texts = [seq[:MAX_INPUT_LEN] + [input_token2idx["<pad>"]] * max(0, MAX_INPUT_LEN - len(seq)) for seq in test_texts]
test_inputs = torch.tensor(test_texts, dtype=torch.long)

predictions = []
model.to(device)  # Ensure the model is on the correct device
idx2token = {idx: token for token, idx in output_token2idx.items()}

with torch.no_grad():
    for inp in test_inputs:
        inp = inp.unsqueeze(0).to(device)  # Add batch dimension
        output = model(inp, torch.zeros((1, 3), dtype=torch.long).to(device), teacher_forcing_ratio=0)  # Generate prediction
        pred_indices = output.argmax(dim=-1)[0].tolist()  # Get all predicted tokens
        pred_label = next((idx2token[idx] for idx in pred_indices if idx2token[idx] not in ["<eos>", "<pad>", "<sos>"]), "unknown")
        print(pred_label)
        predictions.append(pred_label)


ven
epo
urd
hrx
chk
lin
hrx
bqc
kor
ron
ell
lua
xav
guj
pcd
crh
hnj
hrv
sat
bew
aze
mya
sna
que
aze
nyu
fra
kaz
fin
tzo
pcd
wol
haw
nds
tur
sin
fon
fra
sqi
que
dan
qvi
wal
myv
eng
nso
glk
cmn
diq
nnb
iba
zho
ilo
hat
nso
guj
plt
pcd
mwl
pcd
est
cat
crh
bis
xho
por
sin
hrx
ltz
swc
deu
quy
sat
mwl
srn
ksh
kac
slv
yue
aze
xav
nyu
toj
nob
aze
pcd
pms
nds
glg
que
ber
ful
hmo
nor
lat
wol
hrx
sin
hat
fra
tat
diq
hmo
pcd
tat
tat
bam
nnb
csy
crh
aym
kmr
szl
nnb
bre
bih
pcd
lus
gug
glg
nnb
wal
tzo
que
mya
fas
djk
glk
slv
que
amh
mgh
csb
bel
tum
cmn
acr
nch
arz
fij
umb
ary
ewe
nob
zlm
swc
kur
ido
roh
bod
nso
eng
jbo
nav
san
epo
deu
pcd
hat
ori
cbk
epo
quw
mai
hat
zea
snd
hrx
nob
bis
csb
cym
crh
bcl
quy
zsm
san
pnb
plt
tsn
jav
mwl
rap
nob
hbo
bre
ilo
smo
pcd
ajp
hrv
bew
tyv
guc
ber
wol
slv
mlg
tur
mri
fra
ces
swa
mps
slv
cuk
gcf
swe
kaa
nso
lhu
hbs
kaa
nan
roh
ber
ary
mai
sgs
ceb
tam
cuk
tsn
vep
pcd
ace
chk
swc
san
eus
lmo
bjn
que
bel
swa
aka
pcd
bre
mhr
swc
bos
bew
mlt
sah
fas
que
ksh
slv
deu
ell


In [ ]:
test_df["ID"] = range(1, len(test_df) + 1)
test_df["Label"] = predictions
 # Generate sequential IDs starting from 1
print(test_df)
test_df = test_df.drop(columns= ["Text", "Usage"])
test_df.to_csv("test_predictions.csv", index=False)
print("Predictions saved to test_predictions.csv")

          Usage                                               Text      ID  \
0       Private  Hüttwilen el xe on comune del Canton Turgovia ...       1   
1       Private  La leĝo zorgas pri kompenso de nur la plej gra...       2   
2       Private               پک اپ پر اپنے ڈرائیور سے پہلے پہنچیں       3   
3       Private  Mukmu  Ch'itana mukmu icha Butun nisqaqa nisqa...       4   
4       Private  Iwe   lon ena fansoun   lupwen ra aleani än Mo...       5   
...         ...                                                ...     ...   
190562  Private  Seksyon fin travèse à portant  paske tout aran...  190563   
190563  Private          ἐξῆλθεν δὲ εἰς ⸀Ταρσὸν ἀναζητῆσαι Σαῦλον   190564   
190564  Private  Mi yigo’o sugnagrad e Kan Nthothup miyad tabab...  190565   
190565  Private  En dellos ensayos yá teo camentao esos momento...  190566   
190566  Private  Gei goodou ga helekai gi digaula   ‘ Gimaadou ...  190567   

       Label  
0        ven  
1        epo  
2        urd  
3  